<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/Python-Notebook-Banners/Exercise.png"  style="display: block; margin-left: auto; margin-right: auto;";/>
</div>

# Exercise: Variables and variable selection
© ExploreAI Academy

In this exercise, we apply variance thresholding to select features from a dataset.  

## Learning objectives

By the end of this train, you should be able to:
* Perform dummy variable encoding.
* Implement variance thresholding in Python.
* Use a variance threshold to filter out some features in a dataset.

## Exercises

We are provided with the `Crop_yield` dataset that contains various factors that could influence the yield of a particular crop across different regions.

### Import libraries and dataset

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from statsmodels.formula.api import ols

In [3]:
# Load dataset
df= pd.read_csv("https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/Python/Crop_yield.csv")
df.head(5)

,Region,Temperature,Rainfall,Soil_Type,Fertilizer_Usage,Pesticide_Usage,Irrigation,Crop_Variety,Yield
0,East,23.152156,803.362573,Clayey,204.792011,20.767590,1,Variety B,40.316318
1,West,19.382419,571.567670,Sandy,256.201737,49.290242,0,Variety A,26.846639
2,North,27.895890,-8.699637,Loamy,222.202626,25.316121,0,Variety C,-0.323558
3,East,26.741361,897.426194,Loamy,187.984090,17.115362,0,Variety C,45.440871
4,East,19.090286,649.384694,Loamy,110.459549,24.068804,1,Variety B,35.478118


In [4]:
df.shape

(1000, 9)

### Exercise 1

Our dataset contains several categorical features: `Region`, `Soil_Type`, and `Crop_Variety`. 

Use dummy variable encoding to convert these features into a numerical format suitable for model training. Verify the transformation by displaying the first five rows of the modified dataset.

> How has the number of variables in our dataset changed?

In [5]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

In [6]:
df_dummies = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# convert the dummy to 0 and 1
df_dummies = df_dummies.astype(int)

df_dummies.shape

(1000, 13)

In [7]:
# combine the numeric and dummy variable columns
df_combined = pd.concat([df[numeric_cols], df_dummies], axis=1)

df_combined.shape

(1000, 19)

In [8]:
df_combined.columns

Index(['Temperature', 'Rainfall', 'Fertilizer_Usage', 'Pesticide_Usage',
       'Irrigation', 'Yield', 'Temperature', 'Rainfall', 'Fertilizer_Usage',
       'Pesticide_Usage', 'Irrigation', 'Yield', 'Region_North',
       'Region_South', 'Region_West', 'Soil_Type_Loamy', 'Soil_Type_Sandy',
       'Crop_Variety_Variety B', 'Crop_Variety_Variety C'],
      dtype='object')

In [9]:
df_combined.columns = df_combined.columns.str.replace(' ', '_')

In [10]:
df_combined.head()

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_North,Region_South,Region_West,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_Variety_B,Crop_Variety_Variety_C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,23,803,204,20,1,40,0,0,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,19,571,256,49,0,26,0,0,1,0,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,27,-8,222,25,0,0,1,0,0,1,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,26,897,187,17,0,45,0,0,0,1,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,19,649,110,24,1,35,0,0,0,1,0,1,0


In [11]:
df_combined.describe().T

,count,mean,std,min,25%,50%,75%,max
Temperature,1000.0,25.250082,4.979438,8.065931,21.823442,25.429272,28.687443,42.479389
Rainfall,1000.0,498.018579,199.537595,-167.900036,353.178888,494.189894,644.162691,1084.150513
Fertilizer_Usage,1000.0,171.253446,71.697830,50.519129,109.618317,170.676081,232.056661,299.762375
Pesticide_Usage,1000.0,30.014739,11.478614,10.025101,20.345079,30.340789,39.393116,49.993970
Irrigation,1000.0,0.513000,0.500081,0.000000,0.000000,1.000000,1.000000,1.000000
Yield,1000.0,26.263842,10.181351,-8.426900,18.873992,26.324512,33.537327,54.415047
Temperature,1000.0,24.743000,4.990978,8.000000,21.000000,25.000000,28.000000,42.000000
Rainfall,1000.0,497.535000,199.514282,-167.000000,352.750000,493.500000,643.500000,1084.000000
Fertilizer_Usage,1000.0,170.746000,71.712189,50.000000,109.000000,170.000000,232.000000,299.000000
Pesticide_Usage,1000.0,29.516000,11.460064,10.000000,20.000000,30.000000,39.000000,49.000000


In [12]:
df_combined.columns

Index(['Temperature', 'Rainfall', 'Fertilizer_Usage', 'Pesticide_Usage',
       'Irrigation', 'Yield', 'Temperature', 'Rainfall', 'Fertilizer_Usage',
       'Pesticide_Usage', 'Irrigation', 'Yield', 'Region_North',
       'Region_South', 'Region_West', 'Soil_Type_Loamy', 'Soil_Type_Sandy',
       'Crop_Variety_Variety_B', 'Crop_Variety_Variety_C'],
      dtype='object')

In [13]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Temperature             1000 non-null   float64
 1   Rainfall                1000 non-null   float64
 2   Fertilizer_Usage        1000 non-null   float64
 3   Pesticide_Usage         1000 non-null   float64
 4   Irrigation              1000 non-null   int64  
 5   Yield                   1000 non-null   float64
 6   Temperature             1000 non-null   int64  
 7   Rainfall                1000 non-null   int64  
 8   Fertilizer_Usage        1000 non-null   int64  
 9   Pesticide_Usage         1000 non-null   int64  
 10  Irrigation              1000 non-null   int64  
 11  Yield                   1000 non-null   int64  
 12  Region_North            1000 non-null   int64  
 13  Region_South            1000 non-null   int64  
 14  Region_West             1000 non-null   i

### Exercise 2

We want to determine which variables from the new dataset we will use for model training.

Write a function `variance_thresholding` that will use variance thresholding to filter out features based on a variance threshold. The function should accept two parameters, which are the  DataFrame and the threshold value. It should return two DataFrames, one containing only the features that meet the variance threshold criterion, and one containing the scaled DataFrame.

**Hint:** Scaling is crucial as it allows the variance thresholding to be applied uniformly across features. Read up on using the `MinMaxScaler()` function from the `sklearn.preprocessing` package.

In [14]:


def variance_threshold_selector(data, threshold=0.0, selector=VarianceThreshold()) -> tuple:
    """ 
    This function takes a dataframe and a threshold value as input and returns a dataframe with columns 
    that have variance above the threshold.The data should be normalized before applying this function. 
    The threshold value should be between 0 and 1.This function should return two dataframes: 
        one with the selected features and another with the removed features. 
    arguments:
    data: pandas dataframe
        The input dataframe to be filtered. 
    threshold: float
        The threshold value for variance. Columns with variance below this value will be removed.
    selector: VarianceThreshold object
        The VarianceThreshold object to be used for feature selection.
    """
    # initialize the VarianceThreshold object with the given threshold
    selector = selector
    # initialize the MinMaxScaler object
    scaler = MinMaxScaler()

    # fit and transform the data using the scaler
    data_scaled = scaler.fit_transform(data)
    selector.fit(data_scaled)
    selected_features = data_scaled[:, selector.get_support(indices=True)]
    removed_features = data_scaled[:, ~selector.get_support(indices=True)]
    return selected_features, removed_features, selector


### Exercise 3

Using the function we created in **Exercise 2**, apply variance threshold filtering to our encoded dataset, with a threshold of `0.03`. Compare the number of features before and after applying the variance threshold.

In [15]:
threshold = 0.02
selector = VarianceThreshold(threshold)
selected_features, removed_features, selector = variance_threshold_selector(df_combined, threshold=threshold, selector=selector)
selected_features_df = pd.DataFrame(selected_features, columns=df_combined.columns[selector.get_support(indices=True)])
removed_features_df = pd.DataFrame(removed_features, columns=df_combined.columns[~selector.get_support(indices=True)])

selected_features_df.columns, removed_features_df.shape

(Index(['Temperature', 'Rainfall', 'Fertilizer_Usage', 'Pesticide_Usage',
        'Irrigation', 'Yield', 'Temperature', 'Rainfall', 'Fertilizer_Usage',
        'Pesticide_Usage', 'Irrigation', 'Yield', 'Region_North',
        'Region_South', 'Region_West', 'Soil_Type_Loamy', 'Soil_Type_Sandy',
        'Crop_Variety_Variety_B', 'Crop_Variety_Variety_C'],
       dtype='object'),
 (1000, 19))

In [16]:
removed_features_df.columns, removed_features_df.shape

(Index(['Crop_Variety_Variety_C', 'Crop_Variety_Variety_B', 'Soil_Type_Sandy',
        'Soil_Type_Loamy', 'Region_West', 'Region_South', 'Region_North',
        'Yield', 'Irrigation', 'Pesticide_Usage', 'Fertilizer_Usage',
        'Rainfall', 'Temperature', 'Yield', 'Irrigation', 'Pesticide_Usage',
        'Fertilizer_Usage', 'Rainfall', 'Temperature'],
       dtype='object'),
 (1000, 19))

In [17]:
selected_features_df.columns, removed_features_df.shape

(Index(['Temperature', 'Rainfall', 'Fertilizer_Usage', 'Pesticide_Usage',
        'Irrigation', 'Yield', 'Temperature', 'Rainfall', 'Fertilizer_Usage',
        'Pesticide_Usage', 'Irrigation', 'Yield', 'Region_North',
        'Region_South', 'Region_West', 'Soil_Type_Loamy', 'Soil_Type_Sandy',
        'Crop_Variety_Variety_B', 'Crop_Variety_Variety_C'],
       dtype='object'),
 (1000, 19))

### Exercise 4

Train two linear regression models:

**a)** Using all the available features in our dummy encoded dataset from **Exercise 1**.

In [18]:
df_combined.columns, df_combined.shape

(Index(['Temperature', 'Rainfall', 'Fertilizer_Usage', 'Pesticide_Usage',
        'Irrigation', 'Yield', 'Temperature', 'Rainfall', 'Fertilizer_Usage',
        'Pesticide_Usage', 'Irrigation', 'Yield', 'Region_North',
        'Region_South', 'Region_West', 'Soil_Type_Loamy', 'Soil_Type_Sandy',
        'Crop_Variety_Variety_B', 'Crop_Variety_Variety_C'],
       dtype='object'),
 (1000, 19))

In [19]:
# convert df_combined to numeric values only
df_combined = df_combined.apply(pd.to_numeric, errors='coerce')

In [20]:
# remove duplicate columns
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()]

In [21]:
import statsmodels.formula.api as smf

# 1. Construct the formula string directly using list comprehension
features = [col for col in df_combined.columns if col != 'Yield']
formula_str = f"Yield ~ {' + '.join(features)}"

# 2. Fit the OLS model by passing the formula and the entire DataFrame
model = smf.ols(formula=formula_str, data=df_combined).fit()

# 3. Print the summary
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                  Yield   R-squared:                       0.998
Model:                            OLS   Adj. R-squared:                  0.998
Method:                 Least Squares   F-statistic:                 3.464e+04
Date:                Thu, 30 Jul 2026   Prob (F-statistic):               0.00
Time:                        12:42:52   Log-Likelihood:                -716.30
No. Observations:                1000   AIC:                             1459.
Df Residuals:                     987   BIC:                             1522.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  1

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    df_combined.drop(columns=['Yield']), df_combined['Yield'], test_size=0.2, random_state=42)

In [23]:
lm_df_combined = LinearRegression()
lm_df_combined.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [24]:
# using the trained model to make predictions on the training and testing data
y_train_pred = lm_df_combined.predict(X_train)
y_test_pred = lm_df_combined.predict(X_test)

df_test_predictions = pd.DataFrame({'Actual': y_test, 'Predicted': y_test_pred})
df_train_predictions = pd.DataFrame({'Actual': y_train, 'Predicted': y_train_pred})

In [25]:
df_train_predictions.head()

,Actual,Predicted
29,46.286299,45.955342
535,24.775648,24.887273
695,17.201029,17.162702
557,21.642877,21.017586
836,23.855084,23.661714


In [26]:
train_residuals = y_train - y_train_pred
test_residuals = y_test - y_test_pred

In [27]:
# metrics for training data
train_mse = metrics.mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = metrics.r2_score(y_train, y_train_pred)
train_rss = np.sum(train_residuals ** 2)

train_metrics_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'R2', 'RSS'],
    'Value': [train_mse, train_rmse, train_r2, train_rss]
})

train_metrics_df

,Metric,Value
0,MSE,0.242211
1,RMSE,0.492150
2,R2,0.997721
3,RSS,193.769120


In [28]:
df_test_predictions.head()

,Actual,Predicted
521,30.293004,30.079614
737,18.196937,19.023781
740,30.063595,29.204991
660,29.371347,29.180847
411,22.699114,23.082761


In [29]:
# metrics for testing data
test_mse = metrics.mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = metrics.r2_score(y_test, y_test_pred)
test_rss = np.sum(test_residuals ** 2)

test_metrics_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'R2', 'RSS'],
    'Value': [test_mse, test_rmse, test_r2, test_rss]
})
test_metrics_df

,Metric,Value
0,MSE,0.260190
1,RMSE,0.510088
2,R2,0.997191
3,RSS,52.037921


In [30]:
# training and testing metrics comparison add actual values
metrics_comparison_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'R2', 'RSS'],
    'Train': [train_mse, train_rmse, train_r2, train_rss],
    'Test': [test_mse, test_rmse, test_r2, test_rss]
})

metrics_comparison_df

,Metric,Train,Test
0,MSE,0.242211,0.260190
1,RMSE,0.492150,0.510088
2,R2,0.997721,0.997191
3,RSS,193.769120,52.037921


**b)** Using only the features selected through the variance thresholding process in **Exercise 3**.

In [31]:
# using only the selected features for model building
df_new = df_combined[selected_features_df.columns]
df_new.head()

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_North,Region_South,Region_West,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_Variety_B,Crop_Variety_Variety_C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,23.152156,803.362573,204.792011,20.767590,1,40.316318,0,0,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,19.382419,571.567670,256.201737,49.290242,0,26.846639,0,0,1,0,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,1,0,0,1,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,26.741361,897.426194,187.984090,17.115362,0,45.440871,0,0,0,1,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,19.090286,649.384694,110.459549,24.068804,1,35.478118,0,0,0,1,0,1,0


In [32]:
# add Yield column to the new dataframe
# df_new['Yield'] = df_combined['Yield']

In [33]:
# drop duplicate columns
df_new = df_new.loc[:, ~df_new.columns.duplicated()]

In [34]:
# 1. Use the string name of your target column, not the data Series
target_name = df_combined.columns[df_combined.columns == 'Yield'][0]

# 2. Get your selected feature names as a list of strings
feature_names = df_new.columns

# 3. Create the formula string correctly
formula_str_selected = f"{target_name} ~ {' + '.join(feature_names)}"

print(formula_str_selected)


Yield ~ Temperature + Rainfall + Fertilizer_Usage + Pesticide_Usage + Irrigation + Yield + Region_North + Region_South + Region_West + Soil_Type_Loamy + Soil_Type_Sandy + Crop_Variety_Variety_B + Crop_Variety_Variety_C


In [35]:
selected_model = smf.ols(formula=formula_str_selected, data=df_new).fit()
print(selected_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Yield   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 1.998e+31
Date:                Thu, 30 Jul 2026   Prob (F-statistic):               0.00
Time:                        12:42:52   Log-Likelihood:                 30133.
No. Observations:                1000   AIC:                        -6.024e+04
Df Residuals:                     986   BIC:                        -6.017e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept              -2.70

In [36]:
# Train-test split using the new dataframe with selected features
X_train_selected, X_test_selected, y_train_selected, y_test_selected = train_test_split(
    df_new.drop(columns=['Yield']), df_new['Yield'], test_size=0.2, random_state=42)

In [37]:
# Model training and evaluation using the selected features
lm_selected = LinearRegression()

lm_selected.fit(X_train_selected, y_train_selected)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [38]:
# training and testing predictions using the selected features
y_train_pred_selected = lm_selected.predict(X_train_selected)
y_test_pred_selected = lm_selected.predict(X_test_selected)

In [39]:
train_selected_residuals = y_train_selected - y_train_pred_selected
test_selected_residuals = y_test_selected - y_test_pred_selected

In [40]:
# training metrics for the model with selected features
train_selected_mse = metrics.mean_squared_error(y_train_selected, y_train_pred_selected)
train_selected_r2 = metrics.r2_score(y_train_selected, y_train_pred_selected)
train_selected_rss = np.sum(train_selected_residuals ** 2)
train_selected_rmse = np.sqrt(train_selected_mse)

train_selected_metrics_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'R2', 'RSS'],
    'Value': [train_selected_mse, train_selected_rmse, train_selected_r2, train_selected_rss]
})

train_selected_metrics_df

,Metric,Value
0,MSE,0.242211
1,RMSE,0.492150
2,R2,0.997721
3,RSS,193.769120


In [41]:
# Testing metrics for the model with selected features
test_selected_mse = metrics.mean_squared_error(y_test_selected, y_test_pred_selected)
test_selected_r2 = metrics.r2_score(y_test_selected, y_test_pred_selected)
test_selected_rss = np.sum(test_selected_residuals ** 2)
test_selected_rmse = np.sqrt(test_selected_mse)

test_selected_metrics_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'R2', 'RSS'],
    'Value': [test_selected_mse, test_selected_rmse, test_selected_r2, test_selected_rss]
})

test_selected_metrics_df

,Metric,Value
0,MSE,0.260190
1,RMSE,0.510088
2,R2,0.997191
3,RSS,52.037921


In [42]:
# Training and testing metrics comparison for the model with selected features
metrics_selected_comparison_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'R2', 'RSS'],
    'Train': [train_selected_mse, train_selected_rmse, train_selected_r2, train_selected_rss],
    'Test': [test_selected_mse, test_selected_rmse, test_selected_r2, test_selected_rss]
})

metrics_selected_comparison_df

,Metric,Train,Test
0,MSE,0.242211,0.260190
1,RMSE,0.492150,0.510088
2,R2,0.997721,0.997191
3,RSS,193.769120,52.037921


In [43]:
# comparison of metrics between the model with all features and the model with selected features
metrics_comparison_df['Train_Selected'] = metrics_selected_comparison_df['Train']
metrics_comparison_df['Test_Selected'] = metrics_selected_comparison_df['Test']

metrics_comparison_df

,Metric,Train,Test,Train_Selected,Test_Selected
0,MSE,0.242211,0.260190,0.242211,0.260190
1,RMSE,0.492150,0.510088,0.492150,0.510088
2,R2,0.997721,0.997191,0.997721,0.997191
3,RSS,193.769120,52.037921,193.769120,52.037921


## Solutions

**Note:** Use the comments provided to better understand the various parts of the code solutions below.

### Exercise 1

In [ ]:
# Apply dummy variable encoding to the categorical variables
df_encoded = pd.get_dummies(df, columns=["Region", "Soil_Type", "Crop_Variety"], dtype=int)

# Display the first few rows of the modified dataset to confirm the transformation
df_encoded.head()

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_East,Region_North,Region_South,Region_West,Soil_Type_Clayey,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_Variety A,Crop_Variety_Variety B,Crop_Variety_Variety C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,1,0,0,0,1,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,0,0,0,1,0,0,1,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,0,1,0,0,0,1,0,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,1,0,0,0,0,1,0,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,1,0,0,0,0,1,0,0,1,0


In [ ]:
# Check the new number of columns
df_encoded.shape

(1000, 16)

The categorical features have been successfully transformed into numerical format. Each unique value in these columns has been transformed into a separate column with a binary indicator, representing the presence `1` or absence `0` of that category in each row. Note: there has been an update on the `get_dummies` function, and the default output is now True/False.

We examine the new number of columns using the `.shape` attribute. We can see that the columns have increased from `9` to `16`.

### Exercise 2

In [ ]:
def variance_thresholding(df_encoded, threshold_value):
    
   # Splitting the dataset into features and target variable for scaling and training
    X = df_encoded.drop(columns=['Yield']) 
    y = df_encoded['Yield']
    
    # Initialise and fit the scaler to the features only
    scaler = MinMaxScaler()
    scaled_features = scaler.fit_transform(X)
    
    # Convert the scaled features back to a DataFrame
    df_scaled = pd.DataFrame(scaled_features, columns=X.columns)
    
    # Initialise the VarianceThreshold object with the specified threshold value
    selector = VarianceThreshold(threshold=threshold_value)
    
    # Apply the selector to the scaled feature DataFrame
    df_filtered_values = selector.fit_transform(df_scaled)
    
    # Convert the array result into a DataFrame with only the selected features
    df_filtered = pd.DataFrame(df_filtered_values, columns=df_scaled.columns[selector.get_support(indices=True)])
    
    # Return the filtered DataFrame
    return df_filtered, df_scaled

We start by scaling our features using the `MinMaxScaler()`.

We then use the `threshold_value` passed as a parameter to filter out features whose variances fall below this value.

The function eventually returns `df_filtered`, the DataFrame with features whose variances are above the given threshold.

### Exercise 3

In [ ]:
# Call the variance_thresholding() function and pass the given threshold
df_filtered, df_scaled = variance_thresholding(df_encoded, 0.03)

# Compare the number of features before and after variance thresholding
print("Number of features before variance thresholding:", df_scaled.shape[1])
print("Number of features after variance thresholding:", df_filtered.shape[1])  

Number of features before variance thresholding: 15
Number of features after variance thresholding: 13


Using a `0.03` threshold, the number of features has reduced from `15` to `13`, indicating that two of the features have been excluded.

### Exercise 4

**a)**

In [ ]:
X_all = df_encoded.drop(columns=['Yield'])
y = df_encoded['Yield']
# Splitting both datasets into training and testing sets
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(X_all, y, test_size=0.2, random_state=42)

# Training the model using all available features
model_all = LinearRegression()
model_all.fit(X_train_all, y_train_all)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


**b)**

In [ ]:
# Splitting the dataset into training and testing sets
X_train_filtered, X_test_filtered, y_train_filtered, y_test_filtered = train_test_split(df_filtered, y, test_size=0.2, random_state=42)

# Training the model using selected features
model_filtered = LinearRegression()
model_filtered.fit(X_train_filtered, y_train_filtered)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/ExploreAI_logos/EAI_Blue_Dark.png"  style="width:200px";/>
</div>